# Notebook 06: Context-Aware Strategy Generation Engine
**Component 1: AI-Based Business Feasibility Analysis and Personalized Business/Growth Plan Recommendation**

---

## Executive Summary & Research Context
After predicting business feasibility (Phase 3 & 4) and explaining model predictions via SHAP (Phase 5), a passive "Feasible / Infeasible" decision is insufficient for real-world SME decision support. 

This notebook implements **Phase 6: Multi-Objective Strategy Generation Engine**.

### Key Objectives:
1. **Scenario Coverage**: Support both **New Business Proposals** (startups) and **Existing Business Expansions** (growth).
2. **Context & Risk Awareness**: Generate 3 to 4 distinct, structured candidate business/growth strategies based on:
   - Business Category & Stage
   - User Available Capital & Financial Constraints
   - SHAP-identified Negative Risk Factors (e.g., high loan reliance, staff shortage, high competition)
3. **MCDM Criteria Assignment**: Quantify each generated strategy across multiple operational and economic criteria:
   - Initial Investment Required (LKR)
   - Monthly Operating Cost (LKR)
   - Expected Monthly Revenue (LKR)
   - Market Demand & Customer Reach Score (1-100)
   - Competition Mitigation Score (1-5)
   - Resource & Staff Availability Score (1-5)
   - Location Suitability Score (1-5)
   - Implementation Difficulty Score (1-5, Cost criterion)
4. **Strategy Decision Matrix Export**: Save generated strategy matrices into `models/feasibility_model/strategy_matrix_sample.json` for Phase 7 (TOPSIS MCDM Strategy Ranking).


In [1]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
import shap

# Output directory setup
os.makedirs("../models/feasibility_model", exist_ok=True)
os.makedirs("../reports", exist_ok=True)

print("Setup completed successfully.")


C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup completed successfully.


## 1. Load Dataset & Trained Feasibility Model
Loading dataset and model to obtain predictions and local SHAP explanations for strategy generation.


In [2]:
file_path = "../data/raw/Component_One_Core_Feasibility.csv"
df = pd.read_csv(file_path)

y = df["feasibility_label"]
X = df.drop(columns=["feasibility_label", "case_id", "business_description"])

categorical_features = [
    "business_stage", "business_category", "district",
    "province", "location_type", "proposed_action", "competition_level"
]

numerical_features = [
    "available_capital_lkr", "loan_amount_lkr", "monthly_budget_lkr",
    "initial_inventory_cost_lkr", "expected_price_lkr", "expected_customers_per_day",
    "customer_demand_score", "expected_operating_days_per_month",
    "entrepreneur_experience_years", "location_suitability_score",
    "available_staff_count", "required_staff_count",
    "available_equipment_score", "required_equipment_score",
    "supplier_availability_score"
]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

# Train Pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features)
])

random_forest_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'))
])

random_forest_model.fit(X_train, y_train)

trained_preprocessor = random_forest_model.named_steps['preprocessor']
trained_rf = random_forest_model.named_steps['classifier']

feature_names = trained_preprocessor.get_feature_names_out()
X_test_processed = trained_preprocessor.transform(X_test)
X_test_dense = X_test_processed.toarray() if hasattr(X_test_processed, "toarray") else np.array(X_test_processed)

# SHAP Explainer
explainer = shap.TreeExplainer(trained_rf)
shap_explanation = explainer(X_test_dense)
class_names = list(trained_rf.classes_)

print(f"Dataset loaded: {df.shape}")
print(f"Trained RF Classes: {class_names}")


Dataset loaded: (1200, 25)
Trained RF Classes: ['Conditionally Feasible', 'Feasible', 'Infeasible']


## 2. Multi-Objective Strategy Generation Engine Core
Building the `SMEStrategyGenerator` class that analyzes:
1. Business Stage (`New` vs `Existing`)
2. Business Category
3. Financial constraints (`available_capital_lkr`, `loan_amount_lkr`, `monthly_budget_lkr`)
4. Predicted feasibility class & SHAP risk factors

It outputs 3 to 4 structured alternative strategies formatted for TOPSIS ranking.


In [3]:
class SMEStrategyGenerator:
    def __init__(self, feature_names, class_names):
        self.feature_names = feature_names
        self.class_names = class_names
        
    def generate_strategies(self, raw_input_row, predicted_class, class_probs, shap_impacts):
        stage = raw_input_row.get("business_stage", "New")
        category = raw_input_row.get("business_category", "Grocery/Mini-Mart")
        capital = float(raw_input_row.get("available_capital_lkr", 500000))
        loan = float(raw_input_row.get("loan_amount_lkr", 0))
        monthly_budget = float(raw_input_row.get("monthly_budget_lkr", 150000))
        exp_price = float(raw_input_row.get("expected_price_lkr", 500))
        exp_customers = float(raw_input_row.get("expected_customers_per_day", 30))
        demand_score = float(raw_input_row.get("customer_demand_score", 50))
        location_score = float(raw_input_row.get("location_suitability_score", 3))
        comp_level = str(raw_input_row.get("competition_level", "Medium"))
        
        strategies = []
        
        if stage == "New":
            # Strategy 1: Standard Full Physical Setup
            strategies.append({
                "strategy_id": "S1",
                "title": f"Standard Physical Commercial Setup ({category})",
                "description": f"Establish a full-scale physical retail/commercial storefront in {raw_input_row.get('location_type', 'Colombo')}.",
                "strategy_type": "Standard Physical",
                "initial_investment_lkr": round(capital + loan, -3),
                "monthly_operating_cost_lkr": round(monthly_budget, -3),
                "expected_monthly_revenue_lkr": round(exp_customers * exp_price * 26, -3),
                "market_demand_score": min(100.0, demand_score * 1.0),
                "competition_mitigation_score": 3.0 if comp_level == "Medium" else (2.0 if comp_level == "High" else 4.0),
                "resource_availability_score": 3.5,
                "location_suitability_score": location_score,
                "implementation_difficulty": 3.5, # Cost criterion (higher = harder)
                "risk_level": "Medium",
                "shap_risk_addressed": "Provides maximum physical visibility but requires high capital outlay."
            })
            
            # Strategy 2: Lean / Home-Based & Digital Delivery Model
            strategies.append({
                "strategy_id": "S2",
                "title": f"Lean Micro-Setup & Local Delivery ({category})",
                "description": "Start with a low-overhead home/cloud setup leveraging social media, WhatsApp orders, and localized Sri Lankan delivery.",
                "strategy_type": "Lean & Digital",
                "initial_investment_lkr": round(capital * 0.45, -3),
                "monthly_operating_cost_lkr": round(monthly_budget * 0.40, -3),
                "expected_monthly_revenue_lkr": round(exp_customers * exp_price * 22 * 0.7, -3),
                "market_demand_score": min(100.0, demand_score * 0.85),
                "competition_mitigation_score": 4.2,
                "resource_availability_score": 4.5,
                "location_suitability_score": max(3.0, location_score * 0.9),
                "implementation_difficulty": 1.8,
                "risk_level": "Low",
                "shap_risk_addressed": "Mitigates high initial capital shortage and loan dependency by reducing fixed overhead by 55%."
            })
            
            # Strategy 3: Wholesale / B2B Partnership & Pre-Order Model
            strategies.append({
                "strategy_id": "S3",
                "title": f"Pre-Order & Wholesale Supply Partnership ({category})",
                "description": "Partner with established local mini-marts, offices, or wholesale distributors on a pre-order contract basis.",
                "strategy_type": "B2B Partnership",
                "initial_investment_lkr": round(capital * 0.70, -3),
                "monthly_operating_cost_lkr": round(monthly_budget * 0.65, -3),
                "expected_monthly_revenue_lkr": round(exp_customers * exp_price * 26 * 1.1, -3),
                "market_demand_score": min(100.0, demand_score * 1.1),
                "competition_mitigation_score": 3.8,
                "resource_availability_score": 4.0,
                "location_suitability_score": 3.5,
                "implementation_difficulty": 2.8,
                "risk_level": "Low-Medium",
                "shap_risk_addressed": "Bypasses retail location dependence by securing guaranteed B2B order volumes."
            })
            
            # Strategy 4: Shared Co-Working / Kiosk Franchise Model
            strategies.append({
                "strategy_id": "S4",
                "title": f"Shared Retail Kiosk / Pop-Up Franchise ({category})",
                "description": "Rent a micro-kiosk or shared counter space within a high-footfall commercial center or supermarket hub.",
                "strategy_type": "Micro Kiosk",
                "initial_investment_lkr": round(capital * 0.80, -3),
                "monthly_operating_cost_lkr": round(monthly_budget * 0.75, -3),
                "expected_monthly_revenue_lkr": round(exp_customers * exp_price * 26 * 1.2, -3),
                "market_demand_score": min(100.0, demand_score * 1.2),
                "competition_mitigation_score": 3.0,
                "resource_availability_score": 3.8,
                "location_suitability_score": min(5.0, location_score + 1.0),
                "implementation_difficulty": 2.5,
                "risk_level": "Medium",
                "shap_risk_addressed": "Overcomes poor location suitability score by co-locating inside high-demand commercial clusters."
            })
            
        else: # Existing Business Expansion
            # Strategy 1: Direct Capacity & Storefront Expansion
            strategies.append({
                "strategy_id": "S1",
                "title": f"Direct Storefront & Inventory Expansion ({category})",
                "description": "Expand physical floor space, upgrade equipment, and increase inventory capacity to serve higher daily footfall.",
                "strategy_type": "Capacity Expansion",
                "initial_investment_lkr": round(capital + loan, -3),
                "monthly_operating_cost_lkr": round(monthly_budget * 1.35, -3),
                "expected_monthly_revenue_lkr": round(exp_customers * exp_price * 26 * 1.4, -3),
                "market_demand_score": min(100.0, demand_score * 1.25),
                "competition_mitigation_score": 3.5,
                "resource_availability_score": 3.8,
                "location_suitability_score": location_score,
                "implementation_difficulty": 3.2,
                "risk_level": "Medium",
                "shap_risk_addressed": "Capitalizes on existing customer base to maximize revenue throughput."
            })
            
            # Strategy 2: Product Line Diversification & Premium Tier
            strategies.append({
                "strategy_id": "S2",
                "title": f"High-Margin Product Line Diversification ({category})",
                "description": "Introduce premium, high-margin product offerings or complementary services targeting affluent local customer segments.",
                "strategy_type": "Product Diversification",
                "initial_investment_lkr": round(capital * 0.60, -3),
                "monthly_operating_cost_lkr": round(monthly_budget * 0.80, -3),
                "expected_monthly_revenue_lkr": round(exp_customers * exp_price * 1.3 * 26, -3),
                "market_demand_score": min(100.0, demand_score * 1.15),
                "competition_mitigation_score": 4.5,
                "resource_availability_score": 4.2,
                "location_suitability_score": location_score,
                "implementation_difficulty": 2.2,
                "risk_level": "Low-Medium",
                "shap_risk_addressed": "Mitigates high competition by creating a differentiated, higher-margin product category."
            })
            
            # Strategy 3: Digital E-Commerce & Hybrid Delivery Hub
            strategies.append({
                "strategy_id": "S3",
                "title": f"Omnichannel E-Commerce & Subscription Hub ({category})",
                "description": "Launch an online store and recurring subscription delivery model for existing repeat customers in Colombo.",
                "strategy_type": "Omnichannel Digital",
                "initial_investment_lkr": round(capital * 0.40, -3),
                "monthly_operating_cost_lkr": round(monthly_budget * 0.50, -3),
                "expected_monthly_revenue_lkr": round(exp_customers * exp_price * 26 * 1.3, -3),
                "market_demand_score": min(100.0, demand_score * 1.3),
                "competition_mitigation_score": 4.0,
                "resource_availability_score": 4.5,
                "location_suitability_score": max(3.5, location_score),
                "implementation_difficulty": 2.0,
                "risk_level": "Low",
                "shap_risk_addressed": "Expands geographic reach beyond physical location constraints with minimal capital outlay."
            })
            
            # Strategy 4: Second Branch / Express Outlet
            strategies.append({
                "strategy_id": "S4",
                "title": f"Secondary Express Satellite Branch ({category})",
                "description": "Open a compact express satellite outlet in a neighboring high-density suburban sub-district.",
                "strategy_type": "Multi-Branch Growth",
                "initial_investment_lkr": round((capital + loan) * 0.90, -3),
                "monthly_operating_cost_lkr": round(monthly_budget * 1.20, -3),
                "expected_monthly_revenue_lkr": round(exp_customers * exp_price * 26 * 1.5, -3),
                "market_demand_score": min(100.0, demand_score * 1.35),
                "competition_mitigation_score": 3.2,
                "resource_availability_score": 3.2,
                "location_suitability_score": min(5.0, location_score + 0.5),
                "implementation_difficulty": 4.0,
                "risk_level": "Medium-High",
                "shap_risk_addressed": "Drives aggressive growth but requires strict staff and capital management."
            })

        return strategies

generator = SMEStrategyGenerator(feature_names, class_names)
print("SMEStrategyGenerator engine initialized successfully.")


SMEStrategyGenerator engine initialized successfully.


## 3. Test Strategy Generation Across Representative Test Cases
Testing strategy generation for 3 test profiles representing different feasibility predictions and business stages.


In [4]:
# Get test sample predictions
test_preds = trained_rf.predict(X_test_dense)
test_probs = trained_rf.predict_proba(X_test_dense)

# Select 3 sample cases
sample_indices = [0, 5, 10]

generated_scenarios = []

for idx in sample_indices:
    raw_case = X_test.iloc[idx].to_dict()
    case_id = f"TEST_CASE_{idx+1}"
    pred_class = test_preds[idx]
    probs = dict(zip(class_names, [round(float(p), 4) for p in test_probs[idx]]))
    shap_vals = shap_explanation.values[idx, :, :]
    
    strategies = generator.generate_strategies(raw_case, pred_class, probs, shap_vals)
    
    scenario = {
        "case_id": case_id,
        "input_profile": raw_case,
        "prediction": {
            "predicted_class": pred_class,
            "probabilities": probs
        },
        "candidate_strategies": strategies
    }
    generated_scenarios.append(scenario)
    
    print("==================================================")
    print("STRATEGY MATRIX GENERATED FOR " + case_id)
    print("Stage: " + str(raw_case.get('business_stage')) + " | Category: " + str(raw_case.get('business_category')))
    print("Prediction: " + str(pred_class) + " (Probs: " + str(probs) + ")")
    print("Available Capital: LKR " + f"{raw_case.get('available_capital_lkr'):,}" + " | Loan: LKR " + f"{raw_case.get('loan_amount_lkr'):,}")
    print("--------------------------------------------------")
    print("Generated " + str(len(strategies)) + " Candidate Strategies:")
    for st in strategies:
        print("  [" + str(st['strategy_id']) + "] " + str(st['title']))
        print("      Invest: LKR " + f"{st['initial_investment_lkr']:,}" + " | Rev: LKR " + f"{st['expected_monthly_revenue_lkr']:,}" + " | Diff: " + str(st['implementation_difficulty']) + "/5")
        print("      Risk Addressed: " + str(st['shap_risk_addressed']))
        print("")


STRATEGY MATRIX GENERATED FOR TEST_CASE_1
Stage: New | Category: Clothing/ Garment
Prediction: Feasible (Probs: {'Conditionally Feasible': 0.345, 'Feasible': 0.61, 'Infeasible': 0.045})
Available Capital: LKR 366,413 | Loan: LKR 0
--------------------------------------------------
Generated 4 Candidate Strategies:
  [S1] Standard Physical Commercial Setup (Clothing/ Garment)
      Invest: LKR 366,000.0 | Rev: LKR 145,000.0 | Diff: 3.5/5
      Risk Addressed: Provides maximum physical visibility but requires high capital outlay.

  [S2] Lean Micro-Setup & Local Delivery (Clothing/ Garment)
      Invest: LKR 165,000.0 | Rev: LKR 86,000.0 | Diff: 1.8/5
      Risk Addressed: Mitigates high initial capital shortage and loan dependency by reducing fixed overhead by 55%.

  [S3] Pre-Order & Wholesale Supply Partnership (Clothing/ Garment)
      Invest: LKR 256,000.0 | Rev: LKR 159,000.0 | Diff: 2.8/5
      Risk Addressed: Bypasses retail location dependence by securing guaranteed B2B order vo

## 4. Structuring Strategy Decision Matrix for MCDM / TOPSIS (Phase 7)
Converting generated strategies into a clean Pandas Decision Matrix with explicit Benefit vs Cost criteria classification:

### MCDM Criteria Definitions:
1. **Initial Investment (LKR)** → **Cost Criterion** (Lower is better)
2. **Monthly Operating Cost (LKR)** → **Cost Criterion** (Lower is better)
3. **Expected Monthly Revenue (LKR)** → **Benefit Criterion** (Higher is better)
4. **Market Demand Score (1-100)** → **Benefit Criterion** (Higher is better)
5. **Competition Mitigation Score (1-5)** → **Benefit Criterion** (Higher is better)
6. **Resource Availability Score (1-5)** → **Benefit Criterion** (Higher is better)
7. **Location Suitability Score (1-5)** → **Benefit Criterion** (Higher is better)
8. **Implementation Difficulty (1-5)** → **Cost Criterion** (Lower is better)


In [5]:
# Example Decision Matrix for Case 1
sample_case_strategies = generated_scenarios[0]["candidate_strategies"]

matrix_df = pd.DataFrame(sample_case_strategies)[
    ["strategy_id", "title", "initial_investment_lkr", "monthly_operating_cost_lkr", 
     "expected_monthly_revenue_lkr", "market_demand_score", "competition_mitigation_score", 
     "resource_availability_score", "location_suitability_score", "implementation_difficulty"]
]

print("Sample Strategy Decision Matrix (Ready for TOPSIS Phase 7):")
matrix_df


Sample Strategy Decision Matrix (Ready for TOPSIS Phase 7):


,strategy_id,title,initial_investment_lkr,monthly_operating_cost_lkr,expected_monthly_revenue_lkr,market_demand_score,competition_mitigation_score,resource_availability_score,location_suitability_score,implementation_difficulty
0,S1,Standard Physical Commercial Setup (Clothing/ ...,366000.0,72000.0,145000.0,78.0,2.0,3.5,4.0,3.5
1,S2,Lean Micro-Setup & Local Delivery (Clothing/ G...,165000.0,29000.0,86000.0,66.3,4.2,4.5,3.6,1.8
2,S3,Pre-Order & Wholesale Supply Partnership (Clot...,256000.0,47000.0,159000.0,85.8,3.8,4.0,3.5,2.8
3,S4,Shared Retail Kiosk / Pop-Up Franchise (Clothi...,293000.0,54000.0,174000.0,93.6,3.0,3.8,5.0,2.5


## 5. Export Strategy Generation Artifacts
Saving generated strategy metadata to `models/feasibility_model/strategy_matrix_sample.json` for Phase 7 (TOPSIS Ranking).


In [6]:
export_payload = {
    "generated_scenarios": generated_scenarios,
    "criteria_definitions": {
        "initial_investment_lkr": {"type": "cost", "label": "Initial Investment (LKR)"},
        "monthly_operating_cost_lkr": {"type": "cost", "label": "Monthly Operating Cost (LKR)"},
        "expected_monthly_revenue_lkr": {"type": "benefit", "label": "Expected Monthly Revenue (LKR)"},
        "market_demand_score": {"type": "benefit", "label": "Market Demand Score (1-100)"},
        "competition_mitigation_score": {"type": "benefit", "label": "Competition Mitigation Score (1-5)"},
        "resource_availability_score": {"type": "benefit", "label": "Resource Availability Score (1-5)"},
        "location_suitability_score": {"type": "benefit", "label": "Location Suitability Score (1-5)"},
        "implementation_difficulty": {"type": "cost", "label": "Implementation Difficulty (1-5)"}
    }
}

export_path = "../models/feasibility_model/strategy_matrix_sample.json"
with open(export_path, "w", encoding="utf-8") as f:
    json.dump(export_payload, f, indent=4)

print(f"Successfully exported Strategy Generation Artifacts to '{export_path}'")


Successfully exported Strategy Generation Artifacts to '../models/feasibility_model/strategy_matrix_sample.json'
